# Capstone: When Should a Model Decline to Answer?

A classifier that always answers is easy to deploy and hard to trust. The useful version
knows when to stop: it answers when it is confident and passes the case to a person when
it is not.

Building one is simple. Score each input by confidence, choose a threshold, answer above
it and decline below it. That threshold sets your coverage, the fraction of cases the
model handles by itself. Suppose you promise 70 percent coverage. That number is not a
detail, because it decides how many people you need to staff the review desk.

You choose the threshold on the data you have, and it delivers exactly the 70 percent
you promised. Then the data changes. The camera is dirtier this month, the weather is
different, the scanner was replaced.

That is the first question this project asks: does your rule still deliver the coverage
you promised once the data shifts? The second is easy to overlook. You can only calibrate
against shifts you have already seen, so the shifts held back from you are the real test.

Before you continue, make a prediction about the outcome that matters most:

> Compared with calibrating on clean data alone, how much better will a policy that
> accounts for shift hold its promised 70 percent coverage on a corruption it has never
> seen?

The results table will also show what clean calibration alone delivered on those unseen
corruptions. This gives you a useful reference point. If your policy falls short, you can
ask whether the limitation comes from your calibration choice or from the shift itself.

Enter your prediction in the cell below before reading further. One word is sufficient.

In [ ]:
# Choose one:
#
#   "none"    no difference you could measure
#   "small"   a real difference, but too small to act on
#   "large"   big enough to change what you would build
#
# Not graded. The results cell near the end compares your prediction against what the
# run actually found. Being wrong here is the most useful outcome this project offers.

FIRST_CALL = ""

> <font color="#A31F34">**Big picture**</font>
>
> Notice what is being measured. Not accuracy, and not risk, but whether the operating point you promised survives contact with data you did not calibrate on. A rule that quietly answers 53 percent of cases when the contract said 70 has broken the contract even if the answers it does give are fine. Every lab in this course handed you the intervention. Here you author it yourself.

## At a glance

| | |
|---|---|
| Runtime | roughly 5 to 10 minutes on a free Colab GPU runtime, much of it the one-time downloads |
| Data | CIFAR-10 plus a 56 MB subset of the CIFAR-10-C corruption benchmark |
| Model | a frozen ImageNet ResNet-18 with a linear probe |
| You design | the calibration policy, the hypothesis, and the smallest difference worth reporting |
| You submit | seven values the last cell prints, copied to the course page |

## What is graded

The last cell prints seven values and you copy them into the course page. Nothing asks
you to reproduce a number that moves from one run to the next.

| What it covers | When you get it |
|---|---|
| Three analysis helpers you write, run on inputs issued to you | after Task 3 |
| Your proposal: that the design holds, and what it will cost | before anything runs |
| The record: that the run met its contract, and what it spent | after the run |

The probes have definite answers. The helpers are the same whichever path you took, so
switching paths keeps that work, but the inputs are issued per learner and yours differ
from your classmates'. Two further questions ask you to read a result and decide what to
run next; each asks you to select every statement that follows, so there may be more than
one.

Your comparison will land in one of three places, and all three are complete
findings: the effect is at least as large as the one you declared worth acting on, it
is at most that large, or this many seeds cannot tell those apart. Reporting the third
honestly scores as well as reporting the first.

## Setup

The first cell fetches the shared contract layer. It runs in Colab and on a local
machine, and it does nothing if the files are already present.

In [ ]:
# Colab bootstrap: fetch the shared capstone modules if they are not already here.
import hashlib
import os
import urllib.request

REPO = ("https://raw.githubusercontent.com/codey-m/deep_learning/main/"
        "final_project/helpers")
NEEDED = ["project_schema.py", "abstention_adapter.py", "cifar10c_cache.py"]
# Digests of the exact module versions this notebook was built and tested against. A
# file that does not match is a stale copy from an earlier session or a truncated
# download, and either one would fail later in a way that looks like your mistake.
DIGESTS = {
    "abstention_adapter.py": "762ff11dc0dfa6752c7c96b5f6a6d5dc610774b24ed8ef39dec0335965caddb4",
    "cifar10c_cache.py": "bb7f0498452cebc0ec7e61606905d5f95a35873ecda17d3d333b27dd2207bab8",
    "project_schema.py": "41d249621c4b4a0ab151d3355c32763a729b37cf90e760549b4688a95a826bd7",
}


def digest_of(path):
    with open(path, "rb") as handle:
        return hashlib.sha256(handle.read()).hexdigest()


for name in NEEDED:
    if not os.path.exists(name) or digest_of(name) != DIGESTS[name]:
        urllib.request.urlretrieve(f"{REPO}/{name}", name)
    if digest_of(name) != DIGESTS[name]:
        raise RuntimeError(
            f"{name} does not match the version this notebook was tested against. "
            f"Delete it and restart the runtime.")
print("contract layer ready:", ", ".join(NEEDED))

In [ ]:
import math
import statistics
import time

import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, models, transforms

import cifar10c_cache as cifar10c
import project_schema as schema
import abstention_adapter as adapter
from abstention_adapter import AbstentionRule, RuleRegistry, TestSliceLocked
from project_schema import Contrast, ProjectPlan, RunRecord

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu")

MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)

# QUICK_MODE is the graded default and is what the runtime estimate above assumes.
# Turning it off replicates over more seeds, which narrows every noise floor
# without changing the design. That is the remedy when a contrast you care about
# comes out inconclusive: more seeds rather than a softer claim.
QUICK_MODE = True
RESOLUTION = 160
TRAIN_PER_CLASS, EVAL_PER_CLASS = 300, 100
PROBE_EPOCHS = 80
SEEDS = tuple(7960 + i for i in range(10 if QUICK_MODE else 16))

TARGET_COVERAGE = 0.70
SCORE = "max_softmax"
# The shift you may calibrate against, and the three you may not.
DEV_SHIFT = ("defocus_blur", 3)
TEST_SHIFTS = (("contrast", 2), ("contrast", 4), ("fog", 3))
METRIC = "coverage_transfer_error"

print(f"device: {DEVICE}")
if DEVICE.type == "cpu":
    print("\nWARNING: no GPU is available, so this will run on CPU. The runtime above "
          "assumes a GPU and this will take several times longer. In Colab, use "
          "Runtime > Change runtime type > GPU, then run this cell again.")
print(f"declared operating point: {TARGET_COVERAGE:.0%} coverage")
print(f"calibration shift: {DEV_SHIFT[0]} severity {DEV_SHIFT[1]}")
print(f"held-out shifts:   " + ", ".join(f"{n} severity {s}" for n, s in TEST_SHIFTS))

### The corruption benchmark, and checking it is what it says

CIFAR-10-C applies a fixed set of corruptions to the CIFAR-10 test set at five
severities. The full benchmark is 1.1 GB, most of which this project never touches, so
what downloads here is a 56 MB subset holding exactly the four blocks the design uses.

The subset is lossless. Each block decodes byte-for-byte to the slice of the original
archive it came from, and the cell below checks that against a digest rather than
asking you to take it on faith.

The labels are not downloaded at all. Each corruption block holds the CIFAR-10 test
images in their original order, so the labels are just the test targets. That is a
convenient assumption and a dangerous one, because if the reconstruction were
misaligned every number in this project would still look plausible. So it gets checked
too, against a real classifier.

In [ ]:
train_set = datasets.CIFAR10("data", train=True, transform=transforms.ToTensor(),
                             download=True)
test_set = datasets.CIFAR10("data", train=False, transform=transforms.ToTensor(),
                            download=True)
train_targets = torch.as_tensor(train_set.targets)

backbone = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
trunk = nn.Sequential(*list(backbone.children())[:-1]).to(DEVICE).eval()
BACKBONE_PARAMETERS = sum(p.numel() for p in trunk.parameters())
FEATURE_DIM = 512
PROBE_PARAMETERS = FEATURE_DIM * 10 + 10

started = time.time()
print("verifying the cached corruption blocks against the original archive...")
print(cifar10c.self_check())
print(f"  ({time.time() - started:.0f}s)")

In [ ]:
@torch.no_grad()
def embed(images):
    """Frozen backbone features for a batch of float NCHW images in [0, 1]."""
    out = []
    for start in range(0, len(images), 128):
        batch = images[start:start + 128]
        batch = nn.functional.interpolate(batch, size=(RESOLUTION, RESOLUTION),
                                          mode="bilinear", align_corners=False)
        out.append(trunk(((batch - MEAN) / STD).to(DEVICE)).flatten(1).cpu())
    return torch.cat(out)


def classify_for_alignment(images):
    """A quick clean-trained classifier, only used to sanity-check the labels."""
    return probe_for_alignment(embed(images).to(DEVICE)).argmax(1).cpu()


def train_probe(features, labels, seed):
    torch.manual_seed(seed)
    probe = nn.Linear(features.shape[1], 10).to(DEVICE)
    optimizer = torch.optim.Adam(probe.parameters(), lr=1e-2, weight_decay=1e-4)
    inputs, targets = features.to(DEVICE), labels.to(DEVICE)
    for _ in range(PROBE_EPOCHS):
        optimizer.zero_grad()
        nn.functional.cross_entropy(probe(inputs), targets).backward()
        optimizer.step()
    return probe


# A throwaway probe on clean data, purely so the label reconstruction can be checked
# against something that actually classifies.
_align_idx = sorted(torch.randperm(
    50000, generator=torch.Generator().manual_seed(1))[:3000].tolist())
_align_z = embed(torch.stack([train_set[i][0] for i in _align_idx]))
probe_for_alignment = train_probe(_align_z, train_targets[_align_idx], 0)

alignment = cifar10c.verify_alignment(classify_for_alignment, sample=1000)
print("label alignment check:")
for key, value in alignment.items():
    print(f"  {key}: {value}")
if not alignment["aligned"]:
    raise RuntimeError("label reconstruction looks misaligned; stop and investigate")

> <font color="#1D4ED8">**Intuition**</font>
>
> The shuffled control is the part that makes this a real check. Accuracy on the corrupted block being high is only meaningful next to accuracy against deliberately shuffled labels being at chance. One number alone would not tell you whether the alignment was right or whether the task was simply easy.

### Splits, and one thing that is easy to get wrong

Three sets per seed. Training comes from the CIFAR-10 train split. Calibration and
final evaluation both come from the test split, and they hold different images.

That last part is not fussiness. If calibration and evaluation used the same pictures
under different corruptions, you would be measuring transfer across corruptions of the
same images, which is a much narrower claim than transfer to data you have not seen.
It is also very easy to mistake one for the other, so the contract checks that the two
sets of image identities are disjoint.

In [ ]:
def split(seed):
    """Training positions, then two disjoint evaluation sets."""
    generator = torch.Generator().manual_seed(seed)
    train, calibration, final = [], [], []
    test_targets = cifar10c.labels(range(10000))
    for class_id in range(10):
        idx = torch.where(train_targets == class_id)[0]
        train += idx[torch.randperm(len(idx), generator=generator)][
            :TRAIN_PER_CLASS].tolist()
        pos = torch.where(test_targets == class_id)[0]
        pos = pos[torch.randperm(len(pos), generator=generator)]
        calibration += pos[:EVAL_PER_CLASS].tolist()
        final += pos[EVAL_PER_CLASS:2 * EVAL_PER_CLASS].tolist()
    return sorted(train), sorted(calibration), sorted(final)


@torch.no_grad()
def predict_and_score(probe, features):
    """Predicted class and confidence score for each example."""
    probability = probe(features.to(DEVICE)).softmax(dim=1).cpu()
    top = probability.topk(2, dim=1)
    if SCORE == "max_softmax":
        score = top.values[:, 0]
    else:
        score = top.values[:, 0] - top.values[:, 1]
    return top.indices[:, 0], score


_t, _c, _f = split(SEEDS[0])
print(f"per seed: {len(_t)} train, {len(_c)} calibration, {len(_f)} final")
print(f"calibration and final share {len(set(_c) & set(_f))} image identities")

### Before Task 1: your issued inputs

Checkpoint 1 on the course page issues you a parameter count, a step count, a batch
size, and a three-seed table of (baseline, treatment) pairs. They are drawn per learner,
so yours differ from your classmates' and the three values you submit have to come from
your own helpers.

Copy them into the block below. You can leave the zeros for now: every probe below runs
either way, and the fixed probes are what tell you your helpers work.

In [ ]:
# From Checkpoint 1 on the course page. Replace the zeros with your issued values.
ISSUED_PARAMETERS = 0
ISSUED_STEPS = 0
ISSUED_BATCH = 0
ISSUED_TABLE = {
    1: (0.0, 0.0),
    2: (0.0, 0.0),
    3: (0.0, 0.0),
}

issued_ready = ISSUED_PARAMETERS > 0 and ISSUED_TABLE[1] != (0.0, 0.0)
print("issued inputs:", "loaded" if issued_ready else "not filled in yet")

## Your task 1: Scope a run to a compute budget

Every lab so far handed you a compute budget. Estimating one is a skill in itself, and
it is the difference between an experiment that finishes inside a Colab session and
one that dies partway through with nothing to show.

Use the cheapest estimate that tracks real cost: the parameters that do work on
each example, multiplied by the number of examples pushed through. Here the frozen backbone dominates,
because every image is embedded several times over, so its forward passes are counted
alongside the probe's own updates. Return the count in billions, so the numbers stay readable:

$$\text{budget units} = \frac{\text{parameters} \times \text{steps} \times \text{batch size}}{10^9}$$

> <font color="#1D4ED8">**Intuition**</font>
>
> This is a proxy, not a stopwatch. It will not predict wall-clock seconds on a particular GPU, and it is not meant to. What it does is rank designs against each other and catch the one that is a hundred times larger than you thought.

In [ ]:
# STUDENT TASK 1: estimate the cost of a run in budget units.
def compute_budget(trainable_parameters, steps, batch_size):
    """Parameters times examples processed, in billions."""
    # TODO: return trainable_parameters * steps * batch_size, expressed in billions.
    return 0.0

In [ ]:
# Probe 1 (fixed input, definite answer): 35,000 parameters, 250 steps, batch size 50.
# This one never changes, so it tells you whether compute_budget works at all.
probe_budget_value = round(compute_budget(35_000, 250, 50), 4)
budget_probe_contract = abs(probe_budget_value - 0.4375) < 1e-4
print(f"R1 probe: {probe_budget_value} budget units "
      f"({'matches' if budget_probe_contract else 'does not match'} the expected 0.4375)")

# The value you submit comes from your own issued inputs.
if issued_ready:
    issued_budget_value = round(
        compute_budget(ISSUED_PARAMETERS, ISSUED_STEPS, ISSUED_BATCH), 4)
    print(f"R1 to submit: {issued_budget_value} budget units")
else:
    issued_budget_value = None
    print("R1 to submit: fill in the issued block above first")

## Your task 2: Compare two conditions the paired way

You will run each calibration policy under ten random seeds. A seed changes which
images land in each split and how the probe initializes, so the same policy does not
give the same coverage error twice.

The naive comparison takes the mean of one condition and subtracts the mean of the
other. The paired comparison takes the difference within each seed first, then
averages those differences.

With the same seeds in both conditions these two give the identical mean. What changes
is the uncertainty around it. One seed might draw an unusually easy calibration set for
every policy at once, and when you difference within that seed, its easiness cancels.
Differencing the group means leaves it in.

How much pairing buys depends on how much the two conditions actually share, so it
is not a fixed number and is not worth taking on faith. When you read the result you
will see both floors, the paired one and the one you would have got by differencing
the group means, and you can judge the size of the difference on your own run.

In [ ]:
# STUDENT TASK 2: the mean within-seed difference between two conditions.
def paired_difference(treatment_by_seed, reference_by_seed):
    """Both arguments map seed -> metric value. Use only the seeds present in both."""
    seeds = sorted(set(treatment_by_seed) & set(reference_by_seed))
    # TODO: build the list of within-seed differences (treatment minus reference),
    # then return their mean.
    return 0.0

In [ ]:
# Probe 2 (fixed input, definite answer): three seeds, two conditions.
PROBE_TREATMENT = {1: 0.30, 2: 0.28, 3: 0.26}
PROBE_REFERENCE = {1: 0.34, 2: 0.33, 3: 0.29}
probe_paired_value = round(paired_difference(PROBE_TREATMENT, PROBE_REFERENCE), 4)
paired_probe_contract = abs(probe_paired_value - (-0.04)) < 1e-4
print(f"R2 probe: {probe_paired_value} "
      f"({'matches' if paired_probe_contract else 'does not match'} the expected -0.04)")

if issued_ready:
    issued_treatment = {seed: ISSUED_TABLE[seed][1] for seed in (1, 2, 3)}
    issued_reference = {seed: ISSUED_TABLE[seed][0] for seed in (1, 2, 3)}
    issued_paired_value = round(
        paired_difference(issued_treatment, issued_reference), 4)
    print(f"R2 to submit: {issued_paired_value}")
else:
    issued_paired_value = None
    print("R2 to submit: fill in the issued block above first")

## Your task 3: Decide when a difference is big enough to believe

A mean difference on its own settles nothing. Ten seeds is a small sample, and a
difference of three coverage points means one thing when the seed-to-seed spread is
half a point and another thing entirely when the spread is ten points.

The floor is the half-width of a t-interval on the paired differences:

$$\text{floor} = t_{0.975,\, n-1} \cdot \frac{s}{\sqrt{n}}$$

where $s$ is the sample standard deviation of the within-seed differences and $n$ is
how many you have. A difference is resolved when its magnitude exceeds its own floor.
Anything smaller is inside the noise your own seeds produce, and you cannot tell it
apart from zero.

The critical value is supplied. You combine the pieces.

> <font color="#B45309">**Watch out**</font>
>
> Resolving a difference and caring about one are separate questions, and the verdict below combines them into a single call. If the whole interval sits beyond your SESOI, the effect is at least the size you said you would act on. If the whole interval sits inside it, the effect is at most that size, which is a real result and not a failure. If the interval straddles your SESOI, the honest answer is that this many seeds cannot tell, and the fix is more seeds rather than a softer claim.

In [ ]:
# STUDENT TASK 3: the paired noise floor for a list of within-seed differences.
def resolution_floor(differences, critical_value):
    """Half-width of the t-interval around the mean of ``differences``."""
    count = len(differences)
    # TODO: return critical_value * (sample standard deviation) / sqrt(count).
    return 0.0

In [ ]:
# Probe 3 (fixed input, definite answer): the same three seeds as probe 2.
PROBE_DIFFERENCES = [PROBE_TREATMENT[s] - PROBE_REFERENCE[s] for s in (1, 2, 3)]
probe_floor_value = round(
    resolution_floor(PROBE_DIFFERENCES, schema._critical(len(PROBE_DIFFERENCES))), 4)
floor_probe_contract = abs(probe_floor_value - 0.0248) < 1e-4
print(f"R3 probe: {probe_floor_value} "
      f"({'matches' if floor_probe_contract else 'does not match'} the expected 0.0248)")
print(f"the probe difference of {probe_paired_value} is "
      f"{'resolved' if abs(probe_paired_value) > probe_floor_value else 'inside the noise'}")

if issued_ready:
    issued_differences = [ISSUED_TABLE[seed][1] - ISSUED_TABLE[seed][0]
                          for seed in (1, 2, 3)]
    issued_floor_value = round(
        resolution_floor(issued_differences,
                         schema._critical(len(issued_differences))), 4)
    print(f"R3 to submit: {issued_floor_value}")
    print(f"your issued difference of {issued_paired_value} is "
          f"{'resolved' if abs(issued_paired_value) > issued_floor_value else 'inside the noise'}")
else:
    issued_floor_value = None
    print("R3 to submit: fill in the issued block above first")

## Your task 4: Design your calibration policy

This is the part that is yours. Write a function that returns the confidence threshold
your deployed rule will use.

You are given two sets of scores. `clean_scores` come from uncorrupted images.
`dev_shift_scores` come from the one corruption you are permitted to calibrate
against. You are not given anything from the three held-out shifts, and the contract
inspects your function's arguments to confirm you did not ask for them.

The obvious choice is a quantile. If you want 70 percent coverage, take the threshold
that lets through 70 percent of your calibration scores. The decision that is actually
yours is which scores you take that quantile over, because the confidence
distribution moves when the data does, and a threshold fitted to one distribution
lands somewhere else on another.

> <font color="#B45309">**Watch out**</font>
>
> A higher threshold means fewer answers, so lower coverage. If you find your policy delivers far more coverage than the target rather than far less, check the direction of your quantile before concluding anything about shift.

<details style="border:1px solid #e5e7eb;border-radius:8px;padding:10px 14px;background:#f9fafb;color:#111827;margin:14px 0;">
<summary style="cursor:pointer;font-weight:600;">Policies worth considering</summary>
<div style="margin-top:8px;"><ul><li><b>Clean only.</b> The naive baseline and the control. Tune on pristine data and hope it holds.</li><li><b>Development shift only.</b> The supplied comparison. Tune on the one corruption you have, which is closer to deployment than clean data but is still one specific corruption.</li><li><b>A mixture.</b> Pool clean and shifted scores so the quantile is taken over a spread of conditions rather than over one. Worth asking whether the proportions matter.</li><li><b>The more conservative of two.</b> Compute a threshold on each source and take whichever answers less. This trades coverage for safety, so think about whether that is the trade the contract wants.</li><li><b>Aim off-target deliberately.</b> If shift reliably pushes realised coverage in one direction, a policy could calibrate to something other than the target so that it lands on the target after the shift. Defensible, and worth stating plainly if you do it.</li></ul></div>
</details>

In [ ]:
# STUDENT TASK 4: author your calibration policy.
def learner_threshold(clean_scores, dev_shift_scores, target_coverage):
    """Return the confidence threshold your rule will deploy.

    clean_scores:      (N,) confidence scores on uncorrupted calibration images
    dev_shift_scores:  (N,) scores on the same images under the development shift
    target_coverage:   the fraction of cases you promised to answer

    Higher threshold means fewer answers, so lower coverage.
    """
    # TODO: replace this with your own policy. Calibrating on clean scores alone
    # reproduces the control exactly, which the design contract rejects.
    return torch.quantile(clean_scores, 1.0 - target_coverage).item()


MY_POLICY_REASON = ""  # TODO: one sentence, at least 6 words, on why your
                       # calibration source should transfer to an unseen shift.

In [ ]:
def threshold_on_clean(clean_scores, dev_shift_scores, target_coverage):
    """The control: calibrate on pristine data and deploy that threshold unchanged."""
    return torch.quantile(clean_scores, 1.0 - target_coverage).item()


def threshold_on_dev_shift(clean_scores, dev_shift_scores, target_coverage):
    """The supplied baseline: calibrate on the one corruption you are allowed to see."""
    return torch.quantile(dev_shift_scores, 1.0 - target_coverage).item()


POLICIES = {"calibrate_on_clean": threshold_on_clean,
            "calibrate_on_dev_shift": threshold_on_dev_shift,
            "learner_policy": learner_threshold}
# Each policy names the data it actually calibrated on. The contract requires only that
# it was never the held-out shift, so a policy whose whole design is "use clean data"
# should say so rather than mislabel itself to satisfy a check.
CALIBRATION_SOURCE = {
    "calibrate_on_clean": "clean",
    "calibrate_on_dev_shift": f"{DEV_SHIFT[0]}_{DEV_SHIFT[1]}",
    "learner_policy": f"clean+{DEV_SHIFT[0]}_{DEV_SHIFT[1]}",
}

# Does the policy behave like a policy? Checked on three fixed probe pairs rather than
# one, because a single draw cannot tell a control in disguise from a real policy that
# happened to coincide on that draw. The clean source gives the higher threshold on the
# first pair, the shifted source gives it on the second, and they nearly tie on the
# third. A policy passes if it differs from the clean control on at least one pair.
_gen = torch.Generator().manual_seed(3)
PROBE_PAIRS = (
    (torch.rand(500, generator=_gen), torch.rand(500, generator=_gen) * 0.6),
    (torch.rand(500, generator=_gen) * 0.6, torch.rand(500, generator=_gen)),
    (torch.rand(500, generator=_gen), torch.rand(500, generator=_gen)),
)
_yours = [learner_threshold(c, s, TARGET_COVERAGE) for c, s in PROBE_PAIRS]
_clean_t = [threshold_on_clean(c, s, TARGET_COVERAGE) for c, s in PROBE_PAIRS]
_shift_t = [threshold_on_dev_shift(c, s, TARGET_COVERAGE) for c, s in PROBE_PAIRS]

policy_is_well_formed = all(isinstance(t, float) and math.isfinite(t) for t in _yours)
policy_is_novel = policy_is_well_formed and any(
    abs(mine - clean) > 1e-9 for mine, clean in zip(_yours, _clean_t))

print(f"well formed: {policy_is_well_formed}   differs from the clean control: "
      f"{policy_is_novel}")
if policy_is_well_formed:
    print(f"{'probe':<8}{'yours':>10}{'clean-only':>13}{'dev-shift-only':>17}")
    for index, (mine, clean, shift) in enumerate(zip(_yours, _clean_t, _shift_t), 1):
        print(f"  {index:<6}{mine:>10.4f}{clean:>13.4f}{shift:>17.4f}")

## Your task 5: Declare your prediction before you measure

Four declarations, all made before a single threshold is frozen.

**The hypothesis.** What you expect your policy to do to coverage transfer error, and
the mechanism you think would cause it. Coverage transfer error is the absolute gap
between the coverage you promised and the coverage you got, so lower is better.

**The required contrast.** Which comparison your project stands on. Your policy against
clean calibration asks whether accounting for shift at all helps. Your policy against
the development-shift baseline asks the harder question, whether your way of accounting
for it beats the obvious way.

**The smallest effect worth caring about.** In coverage points. Declaring 0.05 says a
five-point improvement in how well the operating point is held would change which
policy you would deploy, and anything smaller would not.

**The budget.** What the whole experiment will cost, from Task 1.

In [ ]:
# STUDENT TASK 5: declare the design.
MY_HYPOTHESIS = ""  # TODO: at least 12 words. What you expect, and the
                    # mechanism you think causes it.

# TODO: which comparison does your project stand on? treatment is always
# "learner_policy"; reference is "calibrate_on_clean" or "calibrate_on_dev_shift";
# direction is "less" (your policy holds coverage better) or "greater" (worse).
REQUIRED_CONTRAST = Contrast("learner_vs_clean", "learner_policy",
                             "calibrate_on_clean", required=True, direction="less")

# TODO: the smallest change in coverage transfer error you would act on.
# Must be above 0 and at most 0.5.
SESOI_ABSOLUTE = 0.05

> <font color="#A31F34">**Big picture**</font>
>
> Declaring the effect size you care about before measuring is what separates an experiment from a search. If you fix it afterwards, you will fix it wherever your result happened to land, and you will have learned nothing you did not already assume.

In [ ]:
CONDITIONS = ("calibrate_on_clean", "calibrate_on_dev_shift", "learner_policy")
SLICES = tuple(f"{name}_{severity}" for name, severity in TEST_SHIFTS)
CONTRASTS = (
    REQUIRED_CONTRAST,
    Contrast("dev_shift_vs_clean", "calibrate_on_dev_shift", "calibrate_on_clean",
             direction="less"),
    Contrast("learner_vs_dev_shift", "learner_policy", "calibrate_on_dev_shift",
             direction="less"),
)
configs = {c: {"calibration_policy": c, "target_coverage": TARGET_COVERAGE,
               "score": SCORE} for c in CONDITIONS}

train_count = TRAIN_PER_CLASS * 10
slice_count = EVAL_PER_CLASS * 10
# Priced from the inventory the design implies: backbone passes over the training
# split, over the calibration split twice (clean and the development shift), and over
# the final split once per unseen shift, plus the probe's own updates.
#
# Twice, not once per policy. All three policies are handed the same two score sets,
# which is cheaper and is also what guarantees they are calibrated on identical data,
# so the only thing separating them is what they do with it.
CALIBRATION_SOURCES = 2
# Priced with your own Task 1 helper rather than a separate expression, so the function
# you wrote is the one that actually does the work here and at R7.
backbone_passes = (train_count + slice_count * CALIBRATION_SOURCES
                   + slice_count * len(TEST_SHIFTS))
projected_budget_units = round(
    (compute_budget(BACKBONE_PARAMETERS, backbone_passes, 1)
     + compute_budget(PROBE_PARAMETERS, PROBE_EPOCHS, train_count)) * len(SEEDS), 4)

plan = ProjectPlan(
    path="abstention_operating_point",
    question=f"Can a calibration policy hold a declared {TARGET_COVERAGE:.0%} coverage "
             f"target when the corruption family and severity change?",
    hypothesis=MY_HYPOTHESIS,
    control="calibrate_on_clean",
    intervention="calibration_policy",
    declared_change="calibration_policy",
    conditions=CONDITIONS,
    evaluation_slices=SLICES,
    seeds=SEEDS,
    compute_budget=projected_budget_units,
    decisions=(f"target={TARGET_COVERAGE}", f"score={SCORE}",
               f"sesoi={SESOI_ABSOLUTE}"),
    condition_kind="categorical",
    contrasts=CONTRASTS,
    seed_varies="data_and_training",
)

BUDGET_MIN, BUDGET_MAX = 0.0, 20000.0

# The generic proposal contract, run now rather than after the experiment. This is the
# same check the execution contract applies to the plan, so a malformed design fails here
# in a second instead of after the run.
proposal = schema.ContractResult()
schema.check_plan(plan, budget_min=BUDGET_MIN, budget_max=BUDGET_MAX, result=proposal)

design_contract, design_report = schema.checklist({
    "your policy returns a finite threshold on every probe": policy_is_well_formed,
    "it differs from the clean control on at least one probe": policy_is_novel,
    "MY_HYPOTHESIS is at least 12 words": len(MY_HYPOTHESIS.split()) >= 12,
    "MY_POLICY_REASON is at least 6 words": len(MY_POLICY_REASON.split()) >= 6,
    "your required contrast is marked required": REQUIRED_CONTRAST.required,
    "its treatment is 'learner_policy'":
        REQUIRED_CONTRAST.treatment == "learner_policy",
    "its reference is 'calibrate_on_clean' or 'calibrate_on_dev_shift'":
        REQUIRED_CONTRAST.reference in ("calibrate_on_clean",
                                        "calibrate_on_dev_shift"),
    "its direction is 'less' or 'greater'":
        REQUIRED_CONTRAST.direction in ("less", "greater"),
    "SESOI_ABSOLUTE is above 0 and at most 0.5": 0.0 < SESOI_ABSOLUTE <= 0.5,
    "your projected budget is above zero (Task 1 is finished)":
        projected_budget_units > 0,
    "the plan passes the generic proposal checks printed below": bool(proposal.passed),
})

print(f"R4 design contract: {design_contract}")
print(design_report)
if not proposal.passed:
    print(proposal.report())
print(f"R5 projected budget: {projected_budget_units} units for "
      f"{len(CONDITIONS)} policies x {len(SEEDS)} seeds x {len(SLICES)} unseen shifts")
print(f"plan digest: {plan.freeze()}")

> <font color="#B45309">**Watch out**</font>
>
> R4 must read 1 before you go on. If it does not, one of the declarations above is still a placeholder, or your policy still calibrates on clean scores alone. Running the experiment on a design that fails its own contract wastes the compute and the result will not be gradeable.

## Phase 1: calibrate, and freeze

This is the mechanic that makes the project a prediction rather than a report.

Every rule is registered with `RuleRegistry` before any held-out shift is touched. The
registry refuses to hand back a rule for a test slice until `unlock_test()` has been
called, and it refuses to accept a new rule afterwards. Registering a threshold also
returns a digest, which gets stamped onto every row that threshold produces, so a
number in the results table can be traced to the exact rule that generated it.

Pre-registration that is only a promise is worth very little. The point here is that
the ordering is enforced by the object, and the cell below demonstrates that by trying
to break it.

In [ ]:
registry = RuleRegistry()

# Try to read a rule before anything has been frozen. This must fail.
try:
    registry.rule_for(SEEDS[0], CONDITIONS[0])
    print("PROBLEM: the registry handed over a rule with nothing frozen")
except (TestSliceLocked, KeyError) as error:
    print(f"locked as expected: {type(error).__name__}")

# And unlocking with no rules registered must fail too.
try:
    registry.unlock_test()
    print("PROBLEM: the registry unlocked the test slice with no rules registered")
except TestSliceLocked as error:
    print(f"locked as expected: {error}")

In [ ]:
started = time.time()
prepared, thresholds_by, method_seconds = {}, {}, {}
measured_budget_units = 0.0

for seed in SEEDS:
    train_idx, calib_idx, final_idx = split(seed)
    train_z = embed(torch.stack([train_set[i][0] for i in train_idx]))
    train_y = train_targets[train_idx]
    probe = train_probe(train_z, train_y, seed)
    measured_budget_units += (
        compute_budget(BACKBONE_PARAMETERS, len(train_idx), 1)
        + compute_budget(PROBE_PARAMETERS, PROBE_EPOCHS, len(train_idx)))

    # The two calibration sources every policy may draw on, and nothing else.
    clean_features = embed(cifar10c.clean_images(calib_idx))
    shift_features = embed(cifar10c.images(*DEV_SHIFT, calib_idx))
    measured_budget_units += compute_budget(BACKBONE_PARAMETERS,
                                            len(calib_idx) * 2, 1)
    _, clean_scores = predict_and_score(probe, clean_features)
    _, shift_scores = predict_and_score(probe, shift_features)

    for condition in CONDITIONS:
        # R5 and R7 price the supplied model. They cannot price learner-authored
        # code, so it is timed instead. Diagnostic only: a wall clock reading is
        # not comparable between one machine and another.
        policy_started = time.perf_counter()
        threshold = POLICIES[condition](clean_scores, shift_scores, TARGET_COVERAGE)
        method_seconds[condition] = (method_seconds.get(condition, 0.0)
                                     + time.perf_counter() - policy_started)
        registry.freeze(AbstentionRule(
            layer="layer4", score=SCORE, target_coverage=TARGET_COVERAGE,
            threshold=threshold, calibrated_on=CALIBRATION_SOURCE[condition],
            plan_hash=plan.freeze()), seed=seed, condition=condition)
        thresholds_by[(condition, seed)] = threshold

    prepared[seed] = (train_idx, calib_idx, final_idx, probe)
    print(f"  seed {seed} calibrated ({time.time() - started:.0f}s elapsed)", flush=True)

print(f"\n{len(registry.rules)} rules frozen across "
      f"{len(SEEDS)} seeds x {len(CONDITIONS)} policies")
print("mean frozen threshold by policy:")
for condition in CONDITIONS:
    values = [thresholds_by[(condition, s)] for s in SEEDS]
    print(f"  {condition:<24} {statistics.mean(values):.4f}")

> <font color="#A31F34">**Big picture**</font>
>
> Every threshold that will ever be applied is now registered, and the plan hash is baked into each one. If you go back and change the plan, the frozen rules stop matching it and the contract says so. That is what stops a design from quietly becoming whatever the results needed it to be.

## Phase 2: unlock, and meet the unseen shifts

Only now does anything touch the held-out corruptions.

In [ ]:
registry.unlock_test()
print("test slice unlocked; no further rules can be frozen")

records = []
for seed in SEEDS:
    train_idx, calib_idx, final_idx, probe = prepared[seed]
    split_id = schema.split_hash(train_idx, calib_idx + final_idx)
    truth = cifar10c.labels(final_idx)
    for shift, slice_name in zip(TEST_SHIFTS, SLICES):
        features = embed(cifar10c.images(*shift, final_idx))
        measured_budget_units += compute_budget(BACKBONE_PARAMETERS,
                                                len(final_idx), 1)
        prediction, score = predict_and_score(probe, features)
        for condition in CONDITIONS:
            rule = registry.rule_for(seed, condition)
            keep = score >= rule.threshold
            realized = keep.float().mean().item()
            risk = ((prediction[keep] != truth[keep]).float().mean().item()
                    if bool(keep.any()) else 0.0)
            stamp = RuleRegistry.digest(rule)
            for name, value in ((METRIC, abs(realized - TARGET_COVERAGE)),
                                ("realized_coverage", realized),
                                ("selective_risk", risk)):
                records.append(RunRecord(
                    condition=condition, seed=seed, split_hash=split_id,
                    config_hash=schema.config_hash(configs[condition]),
                    training_steps=PROBE_EPOCHS,
                    runtime_seconds=method_seconds[condition],
                    metric_name=name, evaluation_slice=slice_name,
                    value=value, provenance=stamp))
    print(f"  seed {seed} evaluated ({time.time() - started:.0f}s elapsed)", flush=True)

measured_budget_units = round(measured_budget_units, 4)
print(f"\n{len(records)} records in {time.time() - started:.0f}s")
print(f"budget projected {projected_budget_units}, measured {measured_budget_units}")

# Freezing after the fact must now be impossible.
try:
    registry.freeze(AbstentionRule(
        layer="layer4", score=SCORE, target_coverage=TARGET_COVERAGE, threshold=0.5,
        calibrated_on="clean", plan_hash=plan.freeze()), seed=SEEDS[0],
        condition=CONDITIONS[0])
    print("PROBLEM: a rule was frozen after the test slice was read")
except TestSliceLocked as error:
    print(f"\nlocked as expected: {error}")

## Read the result

The table below reports, for each policy and each unseen shift, the coverage it
actually delivered and how far that lands from the 70 percent it promised.

Read the coverage column first. The error column is derived from it, and the sign of
the miss tells you something the absolute error hides: a rule that answers too little
is failing safe, and a rule that answers too much is failing open.

In [ ]:
def mean_over(condition, metric, slice_name=None):
    picked = [r.value for r in records if r.condition == condition
              and r.metric_name == metric
              and (slice_name is None or r.evaluation_slice == slice_name)]
    return statistics.mean(picked)


header = f"{'calibration policy':<24}" + "".join(f"{s:>18}" for s in SLICES)
print(header)
for condition in CONDITIONS:
    row = f"{condition:<24}"
    for slice_name in SLICES:
        row += (f"{mean_over(condition, 'realized_coverage', slice_name):>9.3f}"
                f"{mean_over(condition, METRIC, slice_name):>9.3f}")
    print(row)
print(f"{'':<24}" + "".join(f"{'  cov     err':>18}" for _ in SLICES))

report = schema.contrast_report(records, METRIC, SLICES[0], CONTRASTS,
                                sesoi=SESOI_ABSOLUTE)
print()
print(schema.format_contrasts(report, label=f"coverage error on {SLICES[0]}"))

values = schema.paired_values(records, METRIC, SLICES[0])
required_row = next(row for row in report["rows"]
                    if row["name"] == REQUIRED_CONTRAST.name)
own_differences = [values[REQUIRED_CONTRAST.treatment][s]
                   - values[REQUIRED_CONTRAST.reference][s] for s in SEEDS]
own_mean = paired_difference(values[REQUIRED_CONTRAST.treatment],
                             values[REQUIRED_CONTRAST.reference])
own_floor = resolution_floor(own_differences, schema._critical(len(own_differences)))
helpers_agree = (abs(own_mean - required_row["mean"]) < 1e-9
                 and abs(own_floor - required_row["floor"]) < 1e-9)
print(f"\nyour helpers reproduce the report: {helpers_agree}")

# What pairing bought on this run: the same contrast judged by differencing the group
# means instead of within each seed. The factor depends on how much the two conditions
# share, so it is measured here rather than asserted.
_treat = [values[REQUIRED_CONTRAST.treatment][s] for s in SEEDS]
_ref = [values[REQUIRED_CONTRAST.reference][s] for s in SEEDS]
unpaired_floor = schema._critical(len(SEEDS)) * math.sqrt(
    (statistics.stdev(_treat) ** 2 + statistics.stdev(_ref) ** 2) / len(SEEDS))
if own_floor > 0:
    print(f"floor on your required contrast: {unpaired_floor:.4f} unpaired against "
          f"{own_floor:.4f} paired ({unpaired_floor / own_floor:.1f}x)")
    print("  pairing only buys something where the two conditions rise and fall "
          "together across seeds. A factor near 1.0 means yours barely do, which is a "
          "fact about the conditions rather than a mistake, and is worth a sentence in "
          "your caveat.")
else:
    print("floor comparison skipped: your resolution_floor returned 0, so Task 3 is "
          "not finished. Everything below this point depends on it.")

# The call you made at the top, before the notebook had shown you anything, set
# against what the run found. It is judged against your required contrast, which
# is the comparison the opening question asks about unless you changed the
# reference in Task 5. If you did, read this line against the contrast you chose.
first_call = str(globals().get("FIRST_CALL", "")).strip().lower()
if first_call in ("none", "small", "large"):
    called_large = first_call == "large"
    verdict = required_row["verdict"]
    if verdict == "inconclusive":
        outcome = ("this run cannot separate those two possibilities, so your call "
                   "stands untested rather than wrong")
    elif (verdict == "meaningful") == called_large:
        outcome = "the run agrees with you"
    else:
        outcome = ("the run disagrees with you, which is the more interesting of the "
                   "two outcomes and belongs in your record")
    print(f"\nyour first call was {first_call!r} and the effect came out {verdict}: "
          f"{outcome}.")
else:
    print("\nno first call was recorded at the top of the notebook")

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(12, 4))

width = 0.27
for offset, condition in zip((-width, 0.0, width), CONDITIONS):
    axes[0].bar([i + offset for i in range(len(SLICES))],
                [mean_over(condition, "realized_coverage", s) for s in SLICES],
                width, label=condition.replace("calibrate_on_", ""))
axes[0].axhline(TARGET_COVERAGE, color="#111827", ls="--", lw=1.5)
axes[0].text(len(SLICES) - 0.5, TARGET_COVERAGE + 0.01,
             f"promised {TARGET_COVERAGE:.0%}", ha="right", fontsize=8)
axes[0].set_xticks(range(len(SLICES)))
axes[0].set_xticklabels(SLICES, fontsize=8)
axes[0].set_ylabel("coverage actually delivered")
axes[0].set_title("did the operating point hold?")
axes[0].legend(fontsize=8)

names = [row["name"] for row in report["rows"]]
axes[1].barh(names, [row["mean"] for row in report["rows"]], color="#A31F34")
for index, row in enumerate(report["rows"]):
    axes[1].plot([-row["floor"], row["floor"]], [index, index], color="#111827", lw=2)
axes[1].axvline(0, color="#6b7280", lw=1)
# Drawn on both sides. A one-sided band assumes the declared direction, so a learner
# who legitimately predicts the other way would see it on the wrong side.
axes[1].axvspan(-SESOI_ABSOLUTE, SESOI_ABSOLUTE, color="#1D4ED8", alpha=0.12)
axes[1].tick_params(labelsize=8)
axes[1].set_xlabel("paired difference (bar), floor (black), inside blue is below your SESOI")
axes[1].set_title("is the difference bigger than the noise?")
plt.tight_layout()
plt.show()

## Your task 6: Write the record

Six fields. The last two are the ones that decide whether this is science.

`claim` and `evidence` say what you found and the numbers behind it. `caveat` says
what your design could not control. `not_supported` says what a reader might
reasonably conclude from your result that your result does not actually show.
`next_experiment` names the single change you would make next.

Three of these are checked for form, not for quality, and the checks exist because
each names a way the record gets filled in without being thought about. `caveat` has
to name something the design held constant, because that is the boundary of what your
result covers. `resolved_comparisons` has to use the verdicts the analysis actually
returned, so it is written from the output rather than from memory. And
`not_supported` has to say something your claim does not, because pasting the claim
back is not a limitation.

Before you write anything, look at the line the results cell printed comparing your
`FIRST_CALL` against what the run found. If the run disagreed with you, say so plainly:
`claim` reports what happened, and `not_supported` is not the place to quietly rewrite
what you expected. A capstone that changed your mind is worth more than one that
confirmed it.

Write `not_supported` carefully here, because this project has a specific trap in it.
Holding coverage is not the same as being useful. A rule can deliver exactly the 70
percent it promised while answering the wrong 70 percent, and nothing in the primary
metric would notice. Selective risk was recorded alongside coverage; look at it before
you claim your policy is good rather than merely well calibrated.

In [ ]:
# STUDENT TASK 6: the experiment record. Every field must be non-empty.
learner_record = {
    "claim": "",             # TODO: what you found, in one sentence, with the numbers.
    "evidence": "",          # TODO: the measurements that support the claim.
    "resolved_comparisons": "",  # TODO: which contrasts cleared their floor, and by how much.
    "caveat": "",            # TODO: what your design could not control.
    "not_supported": "",     # TODO: what your result does NOT show.
    "next_experiment": "",   # TODO: the single change you would make next.
}

## The contract over what you ran

The checks below are the ones a research group would run before believing its own
result. The ones specific to this path all concern the pre-registration.

Every seed and policy must have registered its own rule. Every test-slice row must
carry the digest of a rule that was actually frozen, so no number can come from a
threshold that was never registered. The development and held-out shifts must genuinely
differ. And the calibration and final image identities must be disjoint, so what is
being measured is transfer to unseen data rather than transfer across corruptions of
the same pictures.

In [ ]:
result = adapter.run_all_checks(
    plan=plan, registry=registry, records=records, configs=configs,
    dev_shift=DEV_SHIFT, test_shift=TEST_SHIFTS[0], record=learner_record,
    budget_min=BUDGET_MIN, budget_max=BUDGET_MAX, metric=METRIC,
    measured_budget=measured_budget_units,
    dev_positions=prepared[SEEDS[0]][1], test_positions=prepared[SEEDS[0]][2],
    expected_slices=SLICES, policies=POLICIES,
    metrics=("realized_coverage", "selective_risk"))

execution_contract, execution_report = schema.checklist({
    "every contract check passed, listed above": bool(result.passed),
    "your own Task 2 and Task 3 helpers reproduce the reported analysis":
        helpers_agree,
    "the held-out shifts were unlocked only after the rules were frozen":
        registry.test_was_evaluated,
    "every seed and policy registered its own rule":
        len(registry.rules) == len(SEEDS) * len(CONDITIONS),
    "your projected budget is above zero, so the comparison below means "
    "something": projected_budget_units > 0,
    "the compute you spent matches what you projected, within 10%":
        projected_budget_units > 0
        and abs(measured_budget_units - projected_budget_units)
        <= 0.10 * projected_budget_units,
})

print(f"contract passed: {result.passed}")
print(result.report())
print(f"\nR6 execution contract: {execution_contract}")
print(execution_report)

## Report values

Run the cell below once every task is complete and the experiment has finished. It
prints seven labelled values, R1 through R7. Copy each into the box with the matching
label on the course page.

R1 to R3 are your three analysis helpers run on the inputs Checkpoint 1 issued you, so
they are yours rather than the cohort's. R4 and R5 are the proposal you had before the
run. R6 and R7 are what the run actually did. The last two checkpoints ask how you read
your result and what you would run next, and are answered on the course page rather than
in the notebook.

In [ ]:
probe_budget = probe_budget_value if budget_probe_contract else -1.0
probe_paired = probe_paired_value if paired_probe_contract else -1.0
probe_floor = probe_floor_value if floor_probe_contract else -1.0

report_values = {
    "R1: compute budget on your issued inputs": issued_budget_value,
    "R2: paired difference on your issued table": issued_paired_value,
    "R3: resolution floor on your issued table": issued_floor_value,
    "R4: design contract": design_contract,
    "R5: projected budget units for your design": projected_budget_units,
    "R6: execution contract": execution_contract,
    "R7: measured budget units your experiment spent": measured_budget_units,
}
# A bare assertion at the end of a full run tells you something is wrong and nothing
# about what. Each line below names the task that produces it, so a failure points
# somewhere you can act.
ready, ready_report = schema.checklist({
    "issued inputs copied from Checkpoint 1 (block above Task 1)": issued_ready,
    "R1 matches its fixed probe (Task 1, compute_budget)":
        abs(probe_budget - 0.4375) < 1e-4,
    "R2 matches its fixed probe (Task 2, paired_difference)":
        abs(probe_paired - (-0.04)) < 1e-4,
    "R3 matches its fixed probe (Task 3, resolution_floor)":
        abs(probe_floor - 0.0248) < 1e-4,
    "R4 design contract passed (Tasks 4 and 5)": design_contract == 1,
    "R6 execution contract passed (the run and Task 6)": execution_contract == 1,
})
if not ready:
    print("Not ready to submit. Nothing is printed below until these pass:")
    print(ready_report)
else:
    print("CAPSTONE REPORT VALUES")
    for label, value in report_values.items():
        print(f"{label}: {value}")